# Распознавание речи

Выберите GPU runtime, укажите `DATASET_URL` и checkpoint в следующей ячейке. Архив должен содержать `audio/` и, если есть эталоны, `transcriptions/`. Выполняйте ячейки по порядку.

Доступны веса шагов 76 000 и 81 500. По умолчанию выбран шаг 76 000 с лучшим dev-other WER.


In [ ]:
DATASET_URL = ""  # @param {type:"string"}
CHECKPOINT_STEP = 76000  # @param [76000, 81500]


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/YohanKIM-creator/ASR.git"
REPO_REF = "main"
WORKSPACE = Path("/content/ASR-demo")

def git_call(arguments):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    subprocess.run(["git", "-c", "credential.helper=", *arguments], env=env, check=True)

if not (WORKSPACE / ".git").is_dir():
    git_call(["clone", REPO_URL, str(WORKSPACE)])
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
actual_origin = subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip()
assert actual_origin == REPO_URL, "Existing workspace belongs to another repository"
subprocess.run(["git", "diff", "--quiet"], check=True)
git_call(["fetch", "origin", REPO_REF])
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], check=True)
KERNEL_PYTHON = sys.executable
subprocess.run([KERNEL_PYTHON, "scripts/setup_colab_env.py"], check=True)
PYTHON = WORKSPACE / ".venv-colab/bin/python"
print("Repository version:", REPO_REF)


In [ ]:
import json

profile = json.loads(subprocess.check_output([str(PYTHON), "scripts/setup_colab_env.py", "--check"], text=True))
assert profile["cuda"], "Select a GPU runtime for the inference demo"
subprocess.run([str(PYTHON), "scripts/download_assets.py", "--checkpoint-step", str(CHECKPOINT_STEP), "--output", "data/resources"], check=True)
CHECKPOINT = Path("data/resources/model.pt")
LM_PATH = Path("data/resources/lm.arpa")
DECODER = "lm"
BEAM_SIZE = 32


Сначала распознаются три записи LibriSpeech dev-clean и рассчитываются WER/CER на этом небольшом примере.


In [ ]:
def infer(data, output):
    command = [str(PYTHON), "inference.py", "datasets.custom.data_dir=" + str(data),
        "inferencer.from_pretrained=" + str(CHECKPOINT), "inferencer.save_path=" + str(output),
        "inferencer.device=cuda", "inferencer.decoder=" + DECODER, "inferencer.beam_size=" + str(BEAM_SIZE)]
    if LM_PATH is not None:
        command.extend(["inferencer.lm_path=" + str(LM_PATH),
            "inferencer.lm_uppercase=" + "true",
            "inferencer.lm_alpha=" + "0.75",
            "inferencer.lm_beta=" + "2.00"])
    subprocess.run(command, check=True)
def metrics(data, output):
    reference = Path(data) / "transcriptions"
    if reference.is_dir() and list(reference.glob("*.txt")):
        subprocess.run([str(PYTHON), "scripts/calc_metrics.py", "--reference-dir", str(reference),
            "--prediction-dir", str(Path(output) / "custom")], check=True)
    else:
        print("No ground truth; predictions:", Path(output) / "custom")
import zipfile

SAMPLE_ARCHIVE = Path("demo/librispeech-sample.zip")
SAMPLE_DIRECTORY = Path("data/demo-speech")
SAMPLE_DIRECTORY.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SAMPLE_ARCHIVE) as sample_archive:
    sample_archive.extractall(SAMPLE_DIRECTORY)
SAMPLE_DATA = SAMPLE_DIRECTORY / "librispeech-demo"
infer(SAMPLE_DATA, Path("outputs/demo-sample"))
metrics(SAMPLE_DATA, Path("outputs/demo-sample"))


Теперь скачивается архив по `DATASET_URL`. При наличии эталонных текстов будут рассчитаны WER/CER.


In [ ]:
assert DATASET_URL, "Enter dataset archive URL in the first cell"
download_output = subprocess.check_output([str(PYTHON), "scripts/download_dataset.py",
    "--url", DATASET_URL, "--output", "data/demo-user"], text=True)
print(download_output)
dataset = Path(download_output.strip().splitlines()[-1])
infer(dataset, Path("outputs/demo-user"))
metrics(dataset, Path("outputs/demo-user"))
print("Predictions:", Path("outputs/demo-user/custom").resolve())
